# M1D DarijaBERT — Stress TestBinary classification: **SAFE** (0) or **OFFENSIVE** (1). Load model, paste messages, get results.

In [ ]:
# ── 1. Setup ────────────────────────────────────────────────────────from google.colab import drivedrive.mount('/content/drive')!pip install -q transformers torch

In [ ]:
# ── 2. Load M1D ─────────────────────────────────────────────────────import torchimport refrom transformers import AutoTokenizer, AutoModelForSequenceClassificationMODEL_PATH = '/content/drive/MyDrive/AEGIS_ML/models/M1D_darijabert/'THRESHOLD = 0.48device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')print(f'Device: {device}')tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH).to(device).eval()print('Done!')

In [ ]:
# ── 3. Normalizer + Predict ─────────────────────────────────────────ARABIZI_TO_ARABIC = {'2': 'ا', '3': 'ع', '5': 'خ', '7': 'ح', '8': 'ق', '9': 'ق'}ARABIZI_DIGRAPHS = {'ch': 'ش', 'gh': 'غ', 'kh': 'خ', 'sh': 'ش', 'th': 'ث', 'dh': 'ذ'}def normalize(text, apply_arabizi=False):    text = str(text).strip()    text = re.sub(r'https?://\S+|www\.\S+', '', text)    text = re.sub(r'@\w+', '', text)    text = re.sub(r'[أإآ]', 'ا', text)    text = text.replace('ى', 'ي').replace('ـ', '')    if apply_arabizi:        for d, l in ARABIZI_DIGRAPHS.items(): text = text.replace(d, l)        for d, l in ARABIZI_TO_ARABIC.items(): text = text.replace(d, l)    return re.sub(r'\s+', ' ', text).strip()def predict(text):    norm = normalize(text, apply_arabizi=any(c in '235789' for c in text))    inputs = tokenizer(norm, return_tensors='pt', truncation=True, max_length=128, padding=True).to(device)    with torch.no_grad():        score = torch.softmax(model(**inputs).logits, dim=-1)[0][1].item()    return {'text': text, 'score': round(score, 4), 'label': 'OFFENSIVE' if score >= THRESHOLD else 'SAFE'}# Sanity checkr = predict('السلام عليكم كيف داير')print(f"Test: {r['label']} (score={r['score']})")

## Paste Your Test MessagesGenerate messages from ChatGPT/Gemini in the cloud, then paste them below.Format: one message per line.  Set `expected` to `'SAFE'` or `'OFFENSIVE'` for that batch.

In [ ]:
# ── 4. PASTE YOUR MESSAGES HERE ─────────────────────────────────────# Each entry: (list_of_messages, expected_label)# Add as many batches as you want!BATCHES = [    # ── Example: replace these with your own messages ──    ([        'السلام عليكم كي داير لاباس',        'salam khoya labas 3lik',        'صباح الخير خويا',        'bonjour ça va mon ami',        'Hello how are you today',    ], 'SAFE'),    ([        'سير تقود يا ولد القحبة',        'nta 7mar w ma 3refti walo',        'wach nta m3a l3eb ola m3aya',    ], 'OFFENSIVE'),    # ── Add more batches below ──    # ([    #     'your message here',    #     'another message',    # ], 'SAFE'),]print(f'Total batches: {len(BATCHES)}')print(f'Total messages: {sum(len(msgs) for msgs, _ in BATCHES)}')

In [ ]:
# ── 5. Run All Tests ────────────────────────────────────────────────results = []for messages, expected in BATCHES:    for text in messages:        r = predict(text)        r['expected'] = expected        r['correct'] = r['label'] == expected        results.append(r)# ── Print Results ───────────────────────────────────────────────print('=' * 90)print(f"{'':>3} {'Score':>7}  {'Predicted':>10}  {'Expected':>10}  {'':>3} Message")print('=' * 90)current_batch = Nonefor i, r in enumerate(results):    batch_key = r['expected']    if batch_key != current_batch:        current_batch = batch_key        print(f"\n── Expected: {batch_key} ──")    mark = '✓' if r['correct'] else '✗'    print(f"  {mark}  {r['score']:.4f}  {r['label']:>10}  {r['expected']:>10}  │ {r['text'][:55]}")# ── Summary ─────────────────────────────────────────────────────tp = sum(1 for r in results if r['expected'] == 'OFFENSIVE' and r['label'] == 'OFFENSIVE')fp = sum(1 for r in results if r['expected'] == 'SAFE' and r['label'] == 'OFFENSIVE')fn = sum(1 for r in results if r['expected'] == 'OFFENSIVE' and r['label'] == 'SAFE')tn = sum(1 for r in results if r['expected'] == 'SAFE' and r['label'] == 'SAFE')precision = tp / (tp + fp) if (tp + fp) > 0 else 0recall = tp / (tp + fn) if (tp + fn) > 0 else 0f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0acc = (tp + tn) / len(results) if results else 0print()print('=' * 50)print(f'  Total:    {len(results)} messages')print(f'  Accuracy: {acc:.1%}')print(f'  Precision:{precision:.3f}')print(f'  Recall:   {recall:.3f}')print(f'  F1:       {f1:.3f}')print(f'  TP={tp}  FP={fp}  FN={fn}  TN={tn}')print('=' * 50)# ── Show Failures ──────────────────────────────────────────────failures = [r for r in results if not r['correct']]if failures:    print(f"\n❌ FAILURES ({len(failures)}):")    for r in failures:        print(f"   {r['label']:>10} (expected {r['expected']:>10}) score={r['score']:.4f} | {r['text'][:60]}")else:    print('\n🎉 All correct!')

In [ ]:
# ── 6. Quick Single Message Test ────────────────────────────────────# Just run this cell with any message to test itpredict('your darija message here')